In [1]:
!pip install ase

In [2]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import linregress
import matplotlib.pyplot as plt
from matplotlib import gridspec
import ase 
from ase import atoms
from ase.neighborlist import neighbor_list
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq


In [12]:
# ================================================================
# SECCIÓN 1 — CONFIGURACIÓN DEL USUARIO
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Esto es lo ÚNICO que usted toca. Son las "perillas" del experimento
#   numérico. Todo lo demás (datos del material, algoritmos) ya está fijo
#   más abajo y no hay que tocarlo.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   Estos valores se usan en la Sección 7 (el barrido) y en la Sección 9
#   (el flujo principal). Defínalos bien acá una sola vez.

# --- Tamaños de supercelda a simular ---
# Para extraer T_c HAY QUE usar varios tamaños: el método del cumulante de
# Binder localiza T_c en el punto donde las curvas de distintos L se cruzan.
# Con un solo L no se puede localizar T_c.
#   - Producción real (preciso, lento):  [12, 18, 24, 36]
#   - Prueba rápida (solo verificar que corre):  [12]
L_list = [6, 12]

# --- Rango de temperaturas del barrido, EN KELVIN ---
# Si no sabe dónde está T_c, corra primero con un rango amplio. La Sección 9
# imprime una estimación de campo medio que le sirve de guía para afinar
# este rango después.
T_min   = 100.0     # K
T_max   = 450    # K
N_temps = 10       # cuántas temperaturas simular entre T_min y T_max

# --- Parámetros del Monte Carlo ---
# CÓMO SE COMBINAN: para CADA temperatura, el programa corre
#   N_term + N_skip * N_meas  barridos en total.
# Con los valores de abajo: 20000 + 10*10000 = 120000 barridos por
# temperatura. Multiplicado por N_temps y por cada tamaño L, eso define
# cuánto tarda la simulación.

# N_term: barridos de TERMALIZACIÓN que se corren al inicio SIN medir, y
# luego se DESCARTAN. Sirven para que el sistema "olvide" su estado inicial
# arbitrario (espines al azar) y llegue al equilibrio térmico a la
# temperatura T. Analogía: esperar a que un café recién hecho se estabilice
# antes de medir su temperatura — si lo mide mientras todavía se enfría,
# obtiene un número equivocado.
#   - Muy chico  -> promedios sesgados (mide configuraciones no-equilibradas).
#   - Muy grande -> solo desperdicia tiempo, no hace daño.
#   - Cerca de T_c la relajación es más lenta: conviene ser generoso.
#   Valor típico: 10^4 a 10^5.
N_term = 2000

# N_skip: cuántos barridos se dejan pasar ENTRE dos medidas consecutivas.
# Dos configuraciones seguidas son casi idénticas (en un barrido solo
# cambian unos pocos espines), así que medir en cada barrido daría datos
# CORRELACIONADOS que repiten la misma información. Esperar N_skip barridos
# asegura que cada medida sea aproximadamente independiente. Analogía:
# sacarle fotos a una nube lenta — 1000 fotos en un segundo son la misma
# foto repetida; mejor esperar entre una y otra.
#   - Muy chico  -> medidas correlacionadas -> subestima los errores.
#   - Muy grande -> desperdicia tiempo (ya eran independientes).
#   Regla: N_skip >= 2 * tau_int (tiempo de autocorrelación, crece cerca de T_c).
N_skip = 5

# N_meas: cuántas MEDIDAS efectivas se recolectan a cada temperatura
# (después de termalizar, espaciadas por N_skip). De estas salen los
# promedios <M^2>, <M^4>, <E>, <E^2>. Más medidas = menor error estadístico:
# la precisión mejora como 1/sqrt(N_meas). Analogía: una encuesta a 10000
# personas es mucho más confiable que a 10.
#   - Muy chico  -> barras de error grandes, el cruce del cumulante sale borroso.
#   - Muy grande -> mejor estadística, pero más tiempo de cómputo.
#   Valor típico: >= 10^4; cerca de T_c apuntar a 10^5 o más.
N_meas = 2000

# --- Reproducibilidad y salida ---

# seed: la "semilla" que inicializa el generador de números aleatorios.
# La computadora NO genera azar verdadero: usa un algoritmo determinista que
# parte de esta semilla. Al fijarla, cada corrida produce EXACTAMENTE la
# misma secuencia de números aleatorios -> resultados reproducibles bit a
# bit (clave para verificar, depurar y publicar). Analogía: barajar un mazo
# con una instrucción de barajado fija — todos obtienen el mismo orden.
# El valor 42 es arbitrario; puede poner cualquier entero.
seed = 42

# Carpeta donde se guardan los resultados (un archivo por cada tamaño L).
output_dir = "./resultados"

In [13]:

# ================================================================
# SECCIÓN 2 — DATOS DEL MATERIAL (vienen del DFT, NO se tocan)
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Acá vive toda la física del material, calculada previamente con DFT.
#   Son constantes del problema. El programa las lee pero el usuario
#   normalmente no las cambia (a menos que cambie de material).
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   La Sección 4 usa 'cell' y 'ti_positions' para construir la red, y usa
#   las distancias d_k + acoplamientos J_k para armar la "libreta de
#   vecinos". El 'epsilon_base' lo usa para el parámetro de orden (Sección 6).

# Constante de Boltzmann en las unidades del problema (meV/K).
# Es el PUENTE entre energía (meV) y temperatura (K). Es el valor físico
# real, no un "1" de convención: aparece explícito en todas las fórmulas
# que mezclan energía con temperatura.
k_B = 0.0861733   # meV/K

# Vectores de red de la celda unidad 2D (Å).
# Las dos primeras filas (a y b) generan el plano del material.
# La tercera fila (c) es la dirección de vacío: NO es periódica.
cell = np.array([[12.5591, -0.0000,  0.000 ],
                 [-3.1397,  5.4382,  0.000 ],
                 [ 0.0000,  0.0000, 23.928 ]])

# Posiciones de los 6 átomos magnéticos de Ti dentro de la celda unidad (Å).
# Cada uno llevará un espín S = +-1 en la simulación.
ti_positions = np.array([[ 1.1238, 2.4778, 15.81],
                         [-0.4358, 5.1792, 15.81],
                         [ 8.9631, 5.1792, 15.81],
                         [ 2.6835, 5.1792, 15.81],
    #                     [ 7.4034, 2.4778, 15.81],
                         [ 5.8437, 5.1792, 15.81]])

# Acoplamientos de canje del DFT (meV), en el orden [J_1, J_2, J_3].
# El SIGNO importa:
#   J > 0 -> favorece espines paralelos     (tendencia ferromagnética)
#   J < 0 -> favorece espines antiparalelos  (tendencia antiferromagnética)
J_meV = np.array([12.0, -3.5, 1.2])

# Distancias (Å) a las que viven esos acoplamientos, [d_1, d_2, d_3].
# IMPORTANTE: verifíquelas con el histograma de distancias real de su red.
d_anillos = np.array([3.12, 5.10, 6.28])

# Tolerancia para decidir si un par está "a la distancia d_k" (Å).
# Debe ser lo bastante ancha para capturar el pico del histograma, pero
# lo bastante angosta para no invadir el pico vecino.
tol = 0.05

# Patrón del estado fundamental: el signo del momento de cada uno de los
# 6 Ti, leído del DFT (+1 = arriba, -1 = abajo). Define contra qué "molde"
# se mide el orden. Para FM serían todos +1; este ejemplo es alternado.
epsilon_base = np.array([+1, -1, +1, -1, +1, -1]) # PREGUNTAR SOBRE ESTE EPSILON BASE

In [14]:
# ================================================================
# SECCIÓN 3 — ESTIMACIÓN DEL RANGO DE TEMPERATURA (CAMPO MEDIO)
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Antes de simular, queremos una idea APROXIMADA de en qué Kelvin está
#   T_c, para no malgastar tiempo simulando temperaturas donde no pasa nada.
#   El campo medio da una estimación rústica (suele sobreestimar por un
#   factor 1.5-2, pero acierta el orden de magnitud).
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   El resultado se imprime en la Sección 9 como sugerencia. Usted lo usa
#   para afinar T_min y T_max de la Sección 1.

def estimar_Tc_campo_medio(z_coordinaciones, J_meV, k_B):
    """
    Estima T_c (en Kelvin) por campo medio:
        T_c^MF = ( sum_k  z_k * |J_k| ) / k_B
    z_coordinaciones: array [z_1, z_2, z_3] (vecinos por anillo).
    Devuelve una temperatura en Kelvin (sobreestimada, solo orientativa:
    el T_c real suele estar entre 0.2 y 0.7 veces este valor en 2D).
    """
    suma_meV = np.sum(z_coordinaciones * np.abs(J_meV))   # energía en meV
    return suma_meV / k_B          

In [15]:
# ================================================================
# SECCIÓN 4 — CONSTRUIR LA RED Y LA "LIBRETA DE VECINOS"
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Acá pasamos de "6 átomos en una celda" a "miles de espines en una red
#   grande con bordes pegados (PBC)". Y construimos la libreta de direcciones:
#   para cada espín, quiénes son sus vecinos y con qué J interactúa con cada
#   uno. Esta libreta es la estructura central que el algoritmo consultará
#   millones de veces; por eso se calcula UNA sola vez al principio.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   La Sección 5 (Metrópolis) usa la libreta para calcular el cambio de
#   energía rápido. El 'epsilon_array' lo usa la Sección 6.

def construir_supercelda(L, cell, ti_positions):
    """
    Replica la celda unidad L x L veces en el plano, con condiciones de
    contorno periódicas (PBC) en a y b, y no-periódica en c. Devuelve un
    objeto que conoce las posiciones de los 6*L*L átomos y la geometría.
    Implementación real: ase.Atoms(..., pbc=[True, True, False]).repeat((L, L, 1)).
    """
    #Construir la celda unidad con los 6 átomos de Ti
    simbolos = ['Ti'] * len(ti_positions) # ASE ya tiene elementos en su librería, aqui le paso 6 atomos de Ti

    #crea la celda base que repetiremos infinitamente
    celda_base = ase.Atoms(
        symbols   = simbolos, #elmento en la red
        positions = ti_positions, #posiciones de los átomos
        cell      = cell, #celda base que creé antes
        pbc       = [True, True, False]  # periódica en a y b, no en c
    )

    #Replica L x L veces en el plano con c intacta
    supercell = celda_base.repeat((L, L, 1))
    return supercell

def construir_libreta_vecinos(supercell, d_anillos, J_meV, tol):
    """
    Recorre todos los pares de espines, calcula la distancia con MÍNIMA
    IMAGEN (respetando PBC; válido para celdas oblicuas porque trabaja en
    coordenadas fraccionarias), y conecta los pares cuya distancia coincida
    con d_1, d_2 o d_3 (dentro de 'tol'), asignándoles J_1, J_2 o J_3.

    Devuelve DOS matrices estáticas de tamaño (N x z_total):
      - neighbor_idx[i] : índices de los vecinos del sitio i
      - neighbor_J[i]   : el J (en meV) de cada uno de esos vecinos
    Implementación real: ase.neighborlist.neighbor_list('ijd', ...).
    """
    i_idx, j_idx, distancias = neighbor_list('ijd', supercell, cutoff=d_anillos[-1] + tol)
    # i es el indice del átomo, j es el índice del atomo vecino, d es la distancia entre ellos
    # supercell es la celda que construí anteriormente
    # cutoff: el corte en el que ASE va a dejar de buscar vecinos más cercanos, tiene cierta tolerancia
    N = len(supercell)

    #Para cada par, determinar a qué anillo pertenece y asignarle J
    pares_por_sitio = [[] for _ in range(N)]
    # loop en el cual conecto los pares para los cuales su distancia concuerde con la distancia d_i con su respectiva J
    for pos in range(len(i_idx)):
        i, j, d = i_idx[pos], j_idx[pos], distancias[pos] # desempaquetamos 
        for (d_k, J_k) in zip(d_anillos, J_meV): # empareja la distancia de los anillos con los J
            if abs(d - d_k) < tol: # condición para que no haya errores numéricos
                pares_por_sitio[i].append((j, J_k))
                break
    #Convertir a matrices numpy estáticas
    z_total = len(pares_por_sitio[0])
    neighbor_idx = np.zeros((N, z_total), dtype=int)
    neighbor_J   = np.zeros((N, z_total))

    #rellena las matrices
    for i, pares in enumerate(pares_por_sitio):
        for col, (j, J_k) in enumerate(pares):
            neighbor_idx[i, col] = j
            neighbor_J[i, col]   = J_k

            
    return neighbor_idx, neighbor_J


def verificar_coordinacion(neighbor_idx, neighbor_J, J_meV):
    """
    Cuenta cuántos vecinos tiene cada sitio a cada distancia (z_1, z_2, z_3
    promedio). DEBEN coincidir con los que asumió el DFT al ajustar los J.
    Si no coinciden, hay un bug en la tolerancia o en la mínima imagen.
    Es la verificación de seguridad MÁS BARATA del flujo entero.
    Devuelve un array [z_1, z_2, z_3].
    """
    # ... cuenta vecinos agrupando por el valor de J ...

    N = neighbor_J.shape[0] # extraemos la dimensión de la matriz 
    z_coordinaciones = np.zeros(len(J_meV)) #creamos la lista para guardar las coordinaciones

    for k, J_k in enumerate(J_meV):
        coincidencias = np.sum(neighbor_J == J_k) # Suma sobre una matriz de booleanos, cada True vale 
                                                  # 1 

        z_coordinaciones[k] = coincidencias / N

        
    
    return z_coordinaciones


def construir_patron_epsilon(L, epsilon_base):
    """
    Replica los 6 signos del estado fundamental (epsilon_base) sobre toda la
    supercelda, generando un array de tamaño N = 6*L*L donde cada espín sabe
    cuál es su orientación "correcta" según el DFT.
    """
    return np.tile(epsilon_base, L * L)   # repite el patrón L*L veces

In [16]:
# ================================================================
# SECCIÓN 5 — ALGORITMO DE METRÓPOLIS (el motor)
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Este es el corazón de la simulación. Genera, una a una, configuraciones
#   de espines representativas del equilibrio térmico a una temperatura T.
#   La regla es simple: pruebo voltear un espín; si baja la energía lo
#   acepto siempre; si la sube, lo acepto con una probabilidad que depende
#   de cuánto sube y de la temperatura.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   La Sección 7 llama a 'metropolis_sweep' miles de veces, primero para
#   termalizar y después para generar las configuraciones que se miden.

def calcular_campo_local(spins, neighbor_idx_i, neighbor_J_i):
    """
    Campo local sobre el sitio i:  h_i = sum_j  J_ij * S_j  (en meV).
    Es la "presión neta" que ejercen los vecinos sobre el sitio i.
    """
    return np.sum(neighbor_J_i * spins[neighbor_idx_i])

def metropolis_step(spins, neighbor_idx, neighbor_J, T, k_B):
    """
    UN intento de voltear un espín al azar (campo externo h = 0).
    delta_E está en meV; T en K; k_B en meV/K -> el exponente del factor
    de Boltzmann es adimensional, como debe ser.
    """
    N = len(spins)
    i = np.random.randint(0, N)                  # elige un sitio i en 0..N-1
    h_i = calcular_campo_local(spins, neighbor_idx[i], neighbor_J[i])
    delta_E = 2.0 * spins[i] * h_i               # cambio de energía (meV)

    if delta_E <= 0.0:
        spins[i] = -spins[i]                     # baja energía -> aceptar siempre
    elif np.random.random() < np.exp(-delta_E / (k_B * T)):
        spins[i] = -spins[i]                     # sube energía -> aceptar con prob de Boltzmann
    # si no se cumple ninguna condición, NO se voltea (se rechaza el intento)

def metropolis_sweep(spins, neighbor_idx, neighbor_J, T, k_B):
    """
    UN barrido = N intentos de volteo. En promedio, cada espín de la red
    recibe una oportunidad de voltearse durante un barrido.
    """
    N = len(spins)
    for _ in range(N):
        metropolis_step(spins, neighbor_idx, neighbor_J, T, k_B)

In [17]:
# ================================================================
# SECCIÓN 6 — MEDICIÓN DE OBSERVABLES
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Dada una configuración de espines, extrae los DOS números que nos
#   interesan: el parámetro de orden (qué tan ordenado está respecto al
#   patrón DFT) y la energía total. De estos saldrán todas las cantidades
#   físicas finales.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   La Sección 7 llama a estas funciones en cada medida y acumula sus
#   potencias (|M|, M^2, M^4, E, E^2) para promediar.

def calcular_M_ord(spins, epsilon_array):
    """
    Parámetro de orden por sitio:  M_ord = (1/N) sum_i  epsilon_i * S_i.
    Cuenta (coincidencias con el patrón) menos (discrepancias), todo sobre N.
    Vale ~+-1 si el sistema está ordenado, ~0 si está desordenado.
    """
    return np.mean(epsilon_array * spins)

def calcular_energia(spins, neighbor_idx, neighbor_J):
    """
    Energía total del sistema (en meV).
    El factor 0.5 corrige el DOBLE CONTEO: cada par (i,j) aparece tanto en
    la libreta de i como en la de j, así que sin el 0.5 contaríamos cada
    interacción dos veces.
    """
    N = len(spins)
    suma = 0.0
    for i in range(N):
        h_i = calcular_campo_local(spins, neighbor_idx[i], neighbor_J[i])
        suma += spins[i] * h_i
    return -0.5 * suma

In [18]:
# ================================================================
# SECCIÓN 7 — BARRIDO EN TEMPERATURA PARA UN TAMAÑO L
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Junta todo lo anterior para un L fijo: construye la red, y luego recorre
#   las temperaturas de MAYOR a MENOR. Esa estrategia se llama "annealing":
#   reusar la configuración final de cada T como inicial de la siguiente
#   acelera muchísimo la termalización (cada T arranca cerca del equilibrio).
#   En cada T, termaliza y mide.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   Devuelve, para cada temperatura, los promedios <|M|>, <M^2>, <M^4>, <E>,
#   <E^2>. La Sección 8 los convierte en cumulante, susceptibilidad y calor
#   específico.

def correr_simulacion(L, cell, ti_positions, d_anillos, J_meV, tol,
                      epsilon_base, T_min, T_max, N_temps,
                      N_term, N_skip, N_meas, k_B):
    """Corre el barrido completo de temperaturas para un tamaño L."""

    # 1) Construir red, libreta de vecinos y patrón epsilon.
    supercell      = construir_supercelda(L, cell, ti_positions)
    nbr_idx, nbr_J = construir_libreta_vecinos(supercell, d_anillos, J_meV, tol)
    epsilon        = construir_patron_epsilon(L, epsilon_base)

    # 2) Verificación de seguridad: z_k debe coincidir con el DFT.
    z_coord = verificar_coordinacion(nbr_idx, nbr_J, J_meV)

    # 3) Inicializar espines al azar (+1 o -1 con probabilidad 50/50).
    N = 6 * L * L
    spins = np.random.choice([-1, +1], size=N)

    # 4) Temperaturas de MAYOR a MENOR (annealing).
    temperaturas = np.linspace(T_max, T_min, N_temps)

    # 5) Barrido en temperatura.
    resultados = {}
    for T in temperaturas:

        # 5a) Termalización: estos barridos NO se miden, se descartan.
        for _ in range(N_term):
            metropolis_sweep(spins, nbr_idx, nbr_J, T, k_B)

        # 5b) Producción: medimos cada N_skip barridos.
        M_abs_acum = 0.0
        M2_acum    = 0.0
        M4_acum    = 0.0
        E_acum     = 0.0
        E2_acum    = 0.0
        for _ in range(N_meas):
            for _ in range(N_skip):
                metropolis_sweep(spins, nbr_idx, nbr_J, T, k_B)
            M = calcular_M_ord(spins, epsilon)
            E = calcular_energia(spins, nbr_idx, nbr_J)
            M_abs_acum += abs(M)
            M2_acum    += M * M
            M4_acum    += M**4
            E_acum     += E
            E2_acum    += E * E

        # 5c) Guardar promedios para esta temperatura.
        resultados[T] = {
            "M_abs": M_abs_acum / N_meas,
            "M2":    M2_acum    / N_meas,
            "M4":    M4_acum    / N_meas,
            "E":     E_acum     / N_meas,
            "E2":    E2_acum    / N_meas,
            "N":     N,
        }

    return resultados, z_coord

In [19]:
# ================================================================
# SECCIÓN 8 — ANÁLISIS: CUMULANTE, chi, C_V Y EXTRACCIÓN DE T_c
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Convierte los promedios crudos en las cantidades físicas. La estrella es
#   el cumulante de Binder: sus curvas para distintos L se cruzan en T_c.
#   Encontrar ese cruce, y extrapolar al sistema infinito, da T_c.
#
# CÓMO SE CONECTA CON LO QUE SIGUE:
#   La Sección 9 llama a estas funciones para obtener el T_c final en K y
#   graficar las curvas.

def calcular_cumulante(M2, M4):
    """Cumulante de Binder:  U_L = 1 - <M^4> / (3 <M^2>^2). Adimensional."""
    return 1.0 - M4 / (3.0 * M2**2)

def calcular_susceptibilidad(M2, M_abs, T, N, k_B):
    """
    Susceptibilidad por sitio (unidades clásicas, con k_B explícito):
        chi = (N / (k_B T)) * (<M^2> - <|M|>^2)
    Se usa <|M|> (valor absoluto) y no <M> porque en fase ordenada el
    sistema salta entre +M_0 y -M_0 y <M> daría cero espuriamente.
    """
    return (N / (k_B * T)) * (M2 - M_abs**2)

def calcular_calor_especifico(E, E2, T, N, k_B):
    """
    Calor específico por sitio (unidades clásicas, con k_B explícito):
        C = (<E^2> - <E>^2) / (N * k_B * T^2)
    """
    return (E2 - E**2) / (N * k_B * T**2)

def encontrar_cruce(curva_L1, curva_L2):
    """
    Dadas las curvas U_L(T) de dos tamaños (diccionarios T -> U), interpola
    ambas con splines y resuelve U_L1(T) = U_L2(T). Devuelve la temperatura
    de cruce (en Kelvin). Esa temperatura es una estimación de T_c todavía
    afectada por tamaño finito.
    Implementación real: scipy.interpolate.CubicSpline + scipy.optimize.brentq.
    """
    # ... interpolación spline + búsqueda de raíz de la diferencia ...
    temps = sorted(curva_L1.keys())

    U_L1 = [curva_L1[T] for T in temps]
    U_L2 = [curva_L2[T] for T in temps]

    print(f"  U_L1: {U_L1}")
    print(f"  U_L2: {U_L2}")


    spline_L1 = CubicSpline(temps, U_L1)
    spline_L2 = CubicSpline(temps, U_L2)

    diferencia = lambda T: spline_L1(T) - spline_L2(T)

    # buscar la raíz en el intervalo [T_min, T_max]
    T_cruce = brentq(diferencia, temps[0], temps[-1])

    return T_cruce

def extrapolar_a_infinito(temperaturas_de_cruce, tamanos):
    """
    Ajusta  T_cruce(L) = T_c(infinito) + a/L  (una recta en 1/L) y devuelve
    la ordenada al origen: T_c del sistema infinito, EN KELVIN.
    Como trabajamos en unidades clásicas, este resultado YA está en Kelvin
    (no hace falta ninguna conversión adicional).
    Implementación real: numpy.polyfit(1/L, T_cruce, deg=1).
    """
    # ... ajuste lineal de T_cruce vs (1/L) ...

    inversos = 1.0 / np.array(tamanos)
    coefs = np.polyfit(inversos, temperaturas_de_cruce, deg=1)

    # la ordenada al origen es T_c del sistema infinito
    T_c_infinito = coefs[1]
    return T_c_infinito

In [20]:
# ================================================================
# SECCIÓN 9 — FLUJO PRINCIPAL (el director de orquesta)
# ================================================================
# ESPÍRITU DE ESTA SECCIÓN:
#   Coordina todo: estima el rango de temperatura, corre la simulación para
#   cada tamaño L, calcula el cumulante, localiza los cruces, extrapola al
#   sistema infinito y reporta T_c EN KELVIN directamente (no hace falta
#   conversión final: ya trabajamos en unidades físicas desde el principio).
#
# CÓMO SE CONECTA CON EL RESTO:
#   Es el punto de entrada. Llama, en orden, a las Secciones 4, 7 y 8.

def main():
    np.random.seed(seed)

    # --- Paso 0: estimación de campo medio para orientar el rango de T ---
    # Construimos rápido la libreta del L más chico solo para sacar los z_k,
    # y con ellos estimamos dónde podría estar T_c.
    sc_chico         = construir_supercelda(L_list[0], cell, ti_positions)
    idx_ch, J_ch     = construir_libreta_vecinos(sc_chico, d_anillos, J_meV, tol)
    z_estimado       = verificar_coordinacion(idx_ch, J_ch, J_meV)
    Tc_mf            = estimar_Tc_campo_medio(z_estimado, J_meV, k_B)
    print(f"Estimacion de campo medio: T_c ~ {Tc_mf:.0f} K (sobreestimada)")
    print(f"Sugerencia: barra T entre ~{0.2*Tc_mf:.0f} K y ~{1.0*Tc_mf:.0f} K")

    # --- Paso 1: correr la simulación para cada tamaño L ---
    todos = {}
    for L in L_list:
        resultados, z_coord = correr_simulacion(
            L, cell, ti_positions, d_anillos, J_meV, tol,
            epsilon_base, T_min, T_max, N_temps,
            N_term, N_skip, N_meas, k_B)
        todos[L] = resultados
        #guardar_a_disco(resultados, f"{output_dir}/L{L}.npz")

    # --- Paso 2: calcular el cumulante U_L(T) para cada L ---
    cumulantes = {}
    for L in L_list:
        cumulantes[L] = {
            T: calcular_cumulante(d["M2"], d["M4"])
            for T, d in todos[L].items()
        }

    # --- Paso 3: localizar los cruces entre pares consecutivos de L ---
    cruces  = []
    pares_L = []
    for L1, L2 in zip(L_list[:-1], L_list[1:]):
        cruces.append(encontrar_cruce(cumulantes[L1], cumulantes[L2]))
        pares_L.append(L1)   # se asocia el cruce al L menor del par

    # --- Paso 4: extrapolar al sistema infinito (resultado YA en Kelvin) ---
    T_c_kelvin = extrapolar_a_infinito(cruces, pares_L)
    print(f"T_c (sistema infinito) = {T_c_kelvin:.1f} K")

    # --- Paso 5: graficar (cumulante, susceptibilidad, calor especifico) ---
    #generar_graficas(todos, cumulantes, T_c_kelvin)


if __name__ == "__main__":
    main()


Estimacion de campo medio: T_c ~ 641 K (sobreestimada)
Sugerencia: barra T entre ~128 K y ~641 K
  U_L1: [np.float64(-221.22222222222226), np.float64(-22.41597796143253), np.float64(-2.673066552709985), np.float64(-0.8514635330855713), np.float64(-0.4026595748838713), np.float64(-0.17352350714596732), np.float64(-0.015535504417179657), np.float64(-0.06008317985998035), np.float64(-0.03326238212536925), np.float64(-0.018171798591849475)]
  U_L2: [np.float64(-82.33333333333336), np.float64(-3.7194008412844806), np.float64(-0.6817542657053173), np.float64(-0.16239319322933166), np.float64(-0.05969199072907805), np.float64(0.021295980582008545), np.float64(-0.05622007721493216), np.float64(-0.029471489809709484), np.float64(0.0014625352723216523), np.float64(-0.08115508296374907)]
T_c (sistema infinito) = 213.6 K


/tmp/ipykernel_12512/2494704543.py:72: RankWarning: Polyfit may be poorly conditioned
  coefs = np.polyfit(inversos, temperaturas_de_cruce, deg=1)
